# WMDP Canonical Script Mini GPU Test

This notebook validates the canonical `llm-unlearn-eco/scripts/evaluate_wmdp.py` entrypoint from a Kaggle-style clean clone. It runs `sample_size=2` for no-corrupt and classifier-free corrupt-hook modes.

Defaults: `PORT_TARGET_CONFIG_NAME=phi-1_5`, `PORT_ATTN_IMPLEMENTATION=eager`, `PORT_TORCH_DTYPE=float16`, `PORT_WMDP_BATCH_SIZE=1`.

In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: f0196e944244c067612e64f818bcd6e9dff50964


In [2]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')

Required packages are already available.


In [3]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This smoke test is intended for Kaggle GPU runtime.')
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')

Python: 3.12.13
Torch: 2.10.0+cu128
CUDA available: True
GPU 0: Tesla T4, VRAM=14.56 GiB
GPU 1: Tesla T4, VRAM=14.56 GiB


## Run Canonical Script

In [4]:
TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH')
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = int(os.environ.get('PORT_WMDP_SAMPLE_SIZE', '2'))
RUN_ORIGINAL = os.environ.get('PORT_RUN_ORIGINAL', '1').strip().lower() not in {'0', 'false', 'no'}
RUN_CORRUPT = os.environ.get('PORT_RUN_CORRUPT', '1').strip().lower() not in {'0', 'false', 'no'}

if IS_KAGGLE:
    print(f'Refreshing repository at {PROJECT_ROOT}')
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'])

SCRIPT_PATH = PROJECT_ROOT / 'llm-unlearn-eco' / 'scripts' / 'evaluate_wmdp.py'
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'
script_source = SCRIPT_PATH.read_text(encoding='utf-8')
required_script_markers = ['--attack_all_prompts', 'SCRIPT_DIR = Path(__file__).resolve().parent']
missing_script_markers = [marker for marker in required_script_markers if marker not in script_source]
if missing_script_markers:
    raise RuntimeError(
        'The cloned repository does not contain the canonical evaluate_wmdp.py yet. '
        f'Missing markers: {missing_script_markers}. Rerun the setup cell or pull/push the latest repo commit.'
    )

SCRIPT_ENV = os.environ.copy()
existing_pythonpath = SCRIPT_ENV.get('PYTHONPATH')
SCRIPT_ENV['PYTHONPATH'] = str(ECO_ROOT) if not existing_pythonpath else str(ECO_ROOT) + os.pathsep + existing_pythonpath
SCRIPT_ENV['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)

def base_command(run_name, task_config):
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        '--model_name', TARGET_CONFIG_NAME,
        '--task_config', task_config,
        '--sample_size', str(SAMPLE_SIZE),
        '--batch_size', str(BATCH_SIZE),
        '--torch_dtype', TORCH_DTYPE,
        '--attn_implementation', ATTN_IMPLEMENTATION,
        '--output_dir', str(OUTPUT_DIR),
        '--run_name', run_name,
    ]
    if TARGET_HF_NAME:
        cmd.extend(['--target_hf_name', TARGET_HF_NAME])
    if TARGET_MODEL_PATH:
        cmd.extend(['--model_path', TARGET_MODEL_PATH])
    return cmd

commands = []
if RUN_ORIGINAL:
    commands.append(base_command(
        f'wmdp_script_mini_original_{TARGET_CONFIG_NAME}',
        'multiple_choice_original.yaml',
    ))
if RUN_CORRUPT:
    cmd = base_command(
        f'wmdp_script_mini_zero_out_{TARGET_CONFIG_NAME}',
        'multiple_choice_zero_out.yaml',
    )
    cmd.append('--attack_all_prompts')
    commands.append(cmd)

for cmd in commands:
    print('\n$ ' + ' '.join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT), env=SCRIPT_ENV)

Refreshing repository at /kaggle/working/PoRT_LLM_Unlearning-Experiment
Already up to date.

$ /usr/bin/python3 /kaggle/working/PoRT_LLM_Unlearning-Experiment/llm-unlearn-eco/scripts/evaluate_wmdp.py --model_name phi-1_5 --task_config multiple_choice_original.yaml --sample_size 2 --batch_size 1 --torch_dtype float16 --attn_implementation eager --output_dir /kaggle/working --run_name wmdp_script_mini_original_phi-1_5


{
  "model_name": "phi-1_5",
  "model_path": null,
  "target_hf_name": "microsoft/phi-1_5",
  "runtime_model_name": "phi-1_5_runtime_wmdp",
  "runtime_config_path": "/kaggle/working/wmdp_script_mini_original_phi-1_5/model_config/phi-1_5_runtime_wmdp.yaml",
  "task_config": "/kaggle/working/PoRT_LLM_Unlearning-Experiment/llm-unlearn-eco/config/task_config/multiple_choice_original.yaml",
  "batch_size": 1,
  "sample_size": 2,
  "seed": 0,
  "torch_dtype": "float16",
  "attn_implementation": "eager",
  "trust_remote_code": null,
  "classifier_threshold": 0.999,
  "classifier_path": null,
  "attack_all_prompts": false,
  "include_baseline": false,
  "include_mmlu": false,
  "use_prefix": false,
  "save_logits": false,
  "run_name": "wmdp_script_mini_original_phi-1_5",
  "run_dir": "/kaggle/working/wmdp_script_mini_original_phi-1_5",
  "tasks": [
    {
      "name": "original",
      "params": {
        "none": null
      }
    }
  ]
}
Loading model: phi-1_5_runtime_wmdp


Loading weights: 100%|██████████| 341/341 [00:00<00:00, 451.33it/s, Materializing param=model.layers.23.self_attn.v_proj.weight]


Number of parameters: 1418270720
Model loaded in 27.18s

Running original_none on wmdp-bio...


Evaluating choice_by_top_logit of wmdp-bio on test: 100%|██████████| 2/2 [00:00<00:00,  2.61it/s]
Evaluating choice_by_top_logit of wmdp-chem on test: 100%|██████████| 2/2 [00:00<00:00, 24.55it/s]


...

Number of parameters: 1418270720
Model loaded in 3.60s

Running corrupt_zero_out_first_n on wmdp-bio...


Evaluating choice_by_top_logit of wmdp-bio on test: 100%|██████████| 2/2 [00:00<00:00,  3.84it/s]
Evaluating choice_by_top_logit of wmdp-chem on test: 100%|██████████| 2/2 [00:00<00:00, 22.63it/s]
Evaluating choice_by_top_logit of wmdp-cyber on test:   0%|          | 0/2 [00:00<?, ?it/s]

{'wmdp-bio_test_choice_by_top_logit': 0.5}
corrupt_zero_out_first_n/wmdp-bio completed in 0.55s with 2 rows.

Running corrupt_zero_out_first_n on wmdp-chem...
{'wmdp-chem_test_choice_by_top_logit': 0.0}
corrupt_zero_out_first_n/wmdp-chem completed in 0.11s with 2 rows.

Running corrupt_zero_out_first_n on wmdp-cyber...


Evaluating choice_by_top_logit of wmdp-cyber on test: 100%|██████████| 2/2 [00:00<00:00, 10.46it/s]


{'wmdp-cyber_test_choice_by_top_logit': 0.5}
corrupt_zero_out_first_n/wmdp-cyber completed in 0.21s with 2 rows.
Wrote /kaggle/working/wmdp_script_mini_zero_out_phi-1_5/run_config.json
Wrote /kaggle/working/wmdp_script_mini_zero_out_phi-1_5/summary.json
Wrote /kaggle/working/wmdp_script_mini_zero_out_phi-1_5/summary_by_run.csv
Wrote /kaggle/working/wmdp_script_mini_zero_out_phi-1_5/predictions.csv


## Validate Artifacts

In [5]:
import pandas as pd

expected_rows = SAMPLE_SIZE * 3
run_names = []
if RUN_ORIGINAL:
    run_names.append(f'wmdp_script_mini_original_{TARGET_CONFIG_NAME}')
if RUN_CORRUPT:
    run_names.append(f'wmdp_script_mini_zero_out_{TARGET_CONFIG_NAME}')

for run_name in run_names:
    run_dir = OUTPUT_DIR / run_name
    summary_path = run_dir / 'summary.json'
    predictions_path = run_dir / 'predictions.csv'
    summary_by_run_path = run_dir / 'summary_by_run.csv'
    for path in [summary_path, predictions_path, summary_by_run_path, run_dir / 'run_config.json']:
        if not path.exists():
            raise FileNotFoundError(path)
    predictions = pd.read_csv(predictions_path)
    if len(predictions) != expected_rows:
        raise AssertionError(f'{run_name}: expected {expected_rows} prediction rows, got {len(predictions)}')
    summary = json.loads(summary_path.read_text())
    print('\n', run_name)
    print('rows:', len(predictions))
    print(pd.read_csv(summary_by_run_path))
    print('summary_overall:', summary.get('summary_overall'))

print('WMDP CANONICAL SCRIPT MINI GPU TEST COMPLETED')


 wmdp_script_mini_original_phi-1_5
rows: 6
       run_label     dataset  count  accuracy
0  original_none    wmdp-bio      2       0.5
1  original_none   wmdp-chem      2       0.0
2  original_none  wmdp-cyber      2       0.0
summary_overall: [{'run_label': 'original_none', 'count': 6, 'accuracy': 0.16666666666666666}]

 wmdp_script_mini_zero_out_phi-1_5
rows: 6
                  run_label     dataset  count  accuracy
0  corrupt_zero_out_first_n    wmdp-bio      2       0.5
1  corrupt_zero_out_first_n   wmdp-chem      2       0.0
2  corrupt_zero_out_first_n  wmdp-cyber      2       0.5
summary_overall: [{'run_label': 'corrupt_zero_out_first_n', 'count': 6, 'accuracy': 0.3333333333333333}]
WMDP CANONICAL SCRIPT MINI GPU TEST COMPLETED
